In [ ]:
import os

# Look two levels up for the root .env file
env_path = os.path.join("..", "..", ".env")

if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                key, value = line.strip().split("=", 1)
                os.environ[key] = value

supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_KEY")

In [22]:
import pandas as pd
from supabase import create_client, Client

SUPABASE_URL = "https://pimvvvtduabnhawaprle.supabase.co"
SUPABASE_KEY = "sb_publishable_hZUCSSpPDUPLbRdwkVnhgQ_p9TPf2O_"

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Supabase connected!")

Supabase connected!


In [23]:
# Print the raw data fetched from Supabase
data = response.data
print("Raw Data:", data)

# Convert to a pandas DataFrame and print as a nice table
df = pd.DataFrame(data)
print(df)

Raw Data: [{'id': 1, 'sale_id': 1, 'invoice_id': 'INV-202301-00001', 'sale_date': '2023-01-10T00:00:00', 'customer_id': 2588, 'product_id': 1274, 'quantity': 2, 'unit_price': 234.79, 'total_price': 469.58, 'channel': 'pood', 'store_location': 'Tallinn', 'payment_method': 'kaart'}, {'id': 2, 'sale_id': 2, 'invoice_id': 'INV-202301-00002', 'sale_date': '2023-01-16T00:00:00', 'customer_id': 4338, 'product_id': 1207, 'quantity': 2, 'unit_price': 241.13, 'total_price': 482.26, 'channel': 'pood', 'store_location': 'Pärnu', 'payment_method': 'järelmaks'}, {'id': 3, 'sale_id': 3, 'invoice_id': 'INV-202301-00003', 'sale_date': '2023-01-05T00:00:00', 'customer_id': 4673, 'product_id': 1264, 'quantity': 1, 'unit_price': 258.46, 'total_price': 221.19, 'channel': 'pood', 'store_location': 'Pärnu', 'payment_method': 'järelmaks'}, {'id': 4, 'sale_id': 4, 'invoice_id': 'INV-202301-00004', 'sale_date': '2023-01-02T00:00:00', 'customer_id': 4677, 'product_id': 1341, 'quantity': 3, 'unit_price': 45.21, '

In [24]:
# 1. Fetch data from Supabase
response = supabase.table('sales').select('*').execute()

# 2. Extract and convert to DataFrame
data = response.data
df = pd.DataFrame(data)

# 3. Print the result
print(df.head())

   id  sale_id        invoice_id            sale_date  customer_id  \
0   1        1  INV-202301-00001  2023-01-10T00:00:00       2588.0   
1   2        2  INV-202301-00002  2023-01-16T00:00:00       4338.0   
2   3        3  INV-202301-00003  2023-01-05T00:00:00       4673.0   
3   4        4  INV-202301-00004  2023-01-02T00:00:00       4677.0   
4   5        5  INV-202301-00005  2023-01-13T00:00:00       2390.0   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1274         2      234.79       469.58    pood        Tallinn   
1        1207         2      241.13       482.26    pood          Pärnu   
2        1264         1      258.46       221.19    pood          Pärnu   
3        1341         3       45.21       135.63    pood          Tartu   
4        1284         1       99.57        99.57    pood          Tartu   

  payment_method  
0          kaart  
1      järelmaks  
2      järelmaks  
3       sularaha  
4          kaart  


In [25]:
import pandas as pd

# 1. Load ALL sales rows from Supabase (bypassing the default 1000 row cap)
sales_response = supabase.table('sales').select('*').range(0, 19999).execute()
df_sales = pd.DataFrame(sales_response.data)

print("=== SALES DATA ===")
print(f"Shape: {df_sales.shape}")
print(df_sales.head())
print("-" * 50)

# 2. Load customer data from Supabase
customers_response = supabase.table('customers').select('*').range(0, 19999).execute()
df_customers = pd.DataFrame(customers_response.data)

print("=== CUSTOMERS DATA ===")
print(f"Shape: {df_customers.shape}")
print(df_customers.head())
print("-" * 50)

# 3. Merge sales and customers on customer_id
df = pd.merge(df_sales, df_customers, on='customer_id', how='left')

# 4. Check quality and structure of merged DataFrame
print("=== MERGED DATASET ===")
print(f"Merged Shape: {df.shape}\n")
print("Data Types:")
print(df.dtypes)
print("\nFirst 5 Rows:")
df.head()

=== SALES DATA ===
Shape: (10118, 12)
   id  sale_id        invoice_id            sale_date  customer_id  \
0   1        1  INV-202301-00001  2023-01-10T00:00:00       2588.0   
1   2        2  INV-202301-00002  2023-01-16T00:00:00       4338.0   
2   3        3  INV-202301-00003  2023-01-05T00:00:00       4673.0   
3   4        4  INV-202301-00004  2023-01-02T00:00:00       4677.0   
4   5        5  INV-202301-00005  2023-01-13T00:00:00       2390.0   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1274         2      234.79       469.58    pood        Tallinn   
1        1207         2      241.13       482.26    pood          Pärnu   
2        1264         1      258.46       221.19    pood          Pärnu   
3        1341         3       45.21       135.63    pood          Tartu   
4        1284         1       99.57        99.57    pood          Tartu   

  payment_method  
0          kaart  
1      järelmaks  
2      järelmaks  
3       sulara

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


In [26]:
import pandas as pd

print("=== STARTING DATA CLEANING ===")
# 1. Print initial shape
initial_rows, initial_cols = df.shape
print(f"Initial shape: {df.shape}")

# 2. Check and remove duplicates (dedupe by invoice_id)
dup_count = df.duplicated(subset=['invoice_id']).sum()
print(f"Duplicates found (by invoice_id): {dup_count}")
df = df.drop_duplicates(subset=['invoice_id'], keep='first')
print(f"Shape after removing duplicates: {df.shape}")

# 3. Check and remove NULL values in critical columns
print("\nNULLs count before cleaning:")
print(df.isnull().sum())

critical_cols = ['customer_id', 'sale_date', 'total_price']
df = df.dropna(subset=critical_cols)

# 4. Parse dates to datetime
df['sale_date'] = pd.to_datetime(df['sale_date'])

# 5. Handle outliers (keep total_price > 0)
outliers_count = (df['total_price'] <= 0).sum()
print(f"\nOutlier rows (total_price <= 0): {outliers_count}")
df = df[df['total_price'] > 0]

# 6. Cleaning Report
print("\n" + "="*30)
print("=== DATA CLEANING REPORT ===")
print("="*30)
print(f"Initial rows: {initial_rows}")
print(f"Final clean rows: {df.shape[0]}")
print(f"Rows removed: {initial_rows - df.shape[0]}")
print(f"Unique customers: {df['customer_id'].nunique()}")
print(f"Date range: {df['sale_date'].min().strftime('%Y-%m-%d')} to {df['sale_date'].max().strftime('%Y-%m-%d')}")
print(f"Final shape: {df.shape}")
print(f"sale_date dtype: {df['sale_date'].dtype}")

=== STARTING DATA CLEANING ===
Initial shape: (10118, 20)
Duplicates found (by invoice_id): 0
Shape after removing duplicates: (10118, 20)

NULLs count before cleaning:
id                      0
sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64

Outlier rows (total_price <= 0): 180

=== DATA CLEANING REPORT ===
Initial rows: 10118
Final clean rows: 8950
Rows removed: 1168
Unique customers: 2540
Date range: 2023-01-01 to 2026-06-28
Final shape: (8950, 20)
sale_date dtype: datetime64[us]


In [27]:
import pandas as pd

print("=== STARTING RFM CUSTOMER SEGMENTATION ===")

# 1. Set reference date
today = pd.to_datetime('2025-02-28')

# 2, 3, 4. Calculate Recency, Frequency, Monetary
rfm = df.groupby('customer_id').agg(
    last_purchase=('sale_date', 'max'),
    frequency=('sale_id', 'count'),
    monetary=('total_price', 'sum')
).reset_index()

# Calculate recency in days
rfm['recency_days'] = (today - rfm['last_purchase']).dt.days

# 5. Assign quintile scores (1-5)
# Note: Recency is REVERSED (lower days = better score = 5)
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency uses rank(method='first') to break ties among identical purchase counts
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Monetary score
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

# 6. Calculate total RFM Score
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# Define segmentation function based on RFM_Score
def assign_segment(score):
    if score >= 13:
        return 'VIP Champions'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'Potential'
    elif score >= 4:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

# Display sample of customer scores
print("=== SAMPLE RFM DATA ===")
print(rfm[['customer_id', 'recency_days', 'frequency', 'monetary', 'RFM_Score', 'Segment']].head())

# Compile summary table (counts and percentage shares)
print("\n" + "="*40)
print("=== RFM SEGMENT SUMMARY TABLE ===")
print("="*40)

summary = rfm['Segment'].value_counts().reset_index()
summary.columns = ['Segment', 'Customer Count']
summary['Share (%)'] = ((summary['Customer Count'] / len(rfm)) * 100).round(2)

# Sort summary by typical priority
segment_order = ['VIP Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']
summary['Segment'] = pd.Categorical(summary['Segment'], categories=segment_order, ordered=True)
summary = summary.sort_values('Segment').reset_index(drop=True)

print(summary.to_string(index=False))

=== STARTING RFM CUSTOMER SEGMENTATION ===
=== SAMPLE RFM DATA ===
   customer_id  recency_days  frequency  monetary  RFM_Score    Segment
0       2001.0            91          2    203.92          6    At Risk
1       2004.0            71          2   1198.56          9  Potential
2       2005.0           148          4    959.60         10      Loyal
3       2006.0           477          1    327.06          3       Lost
4       2007.0            29          1    318.63          7  Potential

=== RFM SEGMENT SUMMARY TABLE ===
      Segment  Customer Count  Share (%)
VIP Champions             455      17.91
        Loyal             679      26.73
    Potential             759      29.88
      At Risk             529      20.83
         Lost             118       4.65


In [28]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [29]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [32]:
import plotly.express as px
import plotly.io as pio

# Set Plotly renderer for VS Code
pio.renderers.default = "notebook_connected"

# Map segment color palette
color_map = {
    'VIP Champions': '#2a9d8f',
    'Loyal': '#e9c46a',
    'Potential': '#f4a261',
    'At Risk': '#e76f51',
    'Lost': '#d62828'
}

# -------------------------------------------------------------------------
# CHART 1: Customer Count by Segment
# -------------------------------------------------------------------------
segment_counts = rfm['Segment'].value_counts().reindex(
    ['VIP Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']
).reset_index()
segment_counts.columns = ['Segment', 'Customer Count']

fig1 = px.bar(
    segment_counts,
    x='Segment',
    y='Customer Count',
    color='Segment',
    color_discrete_map=color_map,
    title='UrbanStyle Customer Count by Segment',
    labels={'Customer Count': 'Number of Customers', 'Segment': 'Customer Segment'},
    text_auto=True
)
fig1.update_layout(showlegend=False, template='plotly_white')
fig1.show()

# -------------------------------------------------------------------------
# CHART 2: Recency vs Monetary Value
# -------------------------------------------------------------------------
fig2 = px.scatter(
    rfm,
    x='recency_days',
    y='monetary',
    color='Segment',
    size='frequency',
    color_discrete_map=color_map,
    title='UrbanStyle Customer Segments: Recency vs Spend',
    labels={
        'recency_days': 'Days Since Last Purchase (Recency)',
        'monetary': 'Total Spend in EUR (Monetary)',
        'frequency': 'Order Count',
        'Segment': 'Segment'
    },
    hover_data=['customer_id']
)
fig2.update_layout(template='plotly_white')
fig2.show()

# -------------------------------------------------------------------------
# CHART 3: Top 10 VIP Champions
# -------------------------------------------------------------------------
top_10_vips = rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary').sort_values('monetary', ascending=True)

fig3 = px.bar(
    top_10_vips,
    x='monetary',
    y='customer_id',
    orientation='h',
    title='Top 10 VIP Champions by Total Spend',
    labels={'monetary': 'Total Spend (EUR)', 'customer_id': 'Customer ID'},
    text_auto='.2f'
)
fig3.update_traces(marker_color='#2a9d8f')
fig3.update_layout(template='plotly_white', yaxis=dict(type='category'))
fig3.show()

# -------------------------------------------------------------------------
# PRINT INSIGHTS
# -------------------------------------------------------------------------
total_rev = rfm['monetary'].sum()
vip_rev = rfm[rfm['Segment'] == 'VIP Champions']['monetary'].sum()
vip_share = (vip_rev / total_rev) * 100
vip_count = len(rfm[rfm['Segment'] == 'VIP Champions'])
at_risk_count = len(rfm[rfm['Segment'] == 'At Risk'])

print("=== DATA INSIGHTS FOR MARKO ===")
print(f"Total Revenue: {total_rev:,.2f} EUR")
print(f"VIP Champions Count: {vip_count} customers")
print(f"VIP Champions Revenue Share: {vip_share:.2f}%")
print(f"At-Risk Customers: {at_risk_count} customers")

=== DATA INSIGHTS FOR MARKO ===
Total Revenue: 2,676,850.54 EUR
VIP Champions Count: 455 customers
VIP Champions Revenue Share: 42.82%
At-Risk Customers: 529 customers


## Business Interpretation for Marko

Our RFM customer segmentation reveals clear revenue concentration and churn risk across UrbanStyle's customer base:

1. **VIP Champions Concentration:** A focused group of VIP Champions generates a disproportionately large share of total revenue. Protecting and retaining these top buyers is critical to maintaining recurring cash flow.
2. **At-Risk Warning:** A significant cohort of previously high-value customers has migrated to the **At Risk** segment due to prolonged inactivity (high recency days).
3. **Growth Potential:** The **Potential** segment displays solid initial order values but lower purchase frequency, representing the primary pool for revenue expansion.

---

## Strategic Recommendations

* **1. Exclusive VIP Loyalty Program:** Launch a high-tier loyalty program for top VIP Champions offering dedicated customer support, early access to new collections, and exclusive private sales to prevent churn among top spenders.
* **2. Automated Win-Back Campaign:** Target the **At Risk** group with personalized win-back email workflows featuring tailored discount codes or "We Miss You" incentives based on their past purchase categories.
* **3. Nurture & Upsell Sequence:** Implement an automated post-purchase onboarding email sequence for **Potential** buyers to encourage their second and third transactions, converting them into long-term **Loyal** customers.

# Advanced RFM Segmentation (Weighted & Granular)

## Methodology Overview
To give Marko and the marketing team more precise, actionable insights, we implement an **Advanced Tiered RFM Model**:
1. **Weighted RFM Scoring:** Monetary value ($M$) is weighted **2x** ($R + F + 2 \times M$) because total spending directly impacts UrbanStyle's bottom line.
2. **Granular Segmentation:** We expand from 5 to **6 granular segments** (adding *Regular Customers* and *New Customers*).
3. **Actionable Marketing Mapping:** Every customer is mapped to a specific strategic campaign based on their score tier.
4. **Data Export:** Automated export to `rfm_segments.csv` for seamless integration into marketing automation tools.

### Advanced Segmentation & Action Mapping
| Weighted RFM Score | Segment | Marketing Action |
| :---: | :--- | :--- |
| **13 – 15** | **VIP Champions** | Early access, VIP discount codes |
| **11 – 12** | **Loyal Customers** | Loyalty programme, rewards |
| **9 – 10** | **Regular Customers** | Cross-sell campaigns |
| **7 – 8** | **New Customers** | Onboarding, welcome series |
| **5 – 6** | **At Risk** | Win-back campaign, personalised email |
| **3 – 4** | **Lost** | Last attempt, big discount |

In [34]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# 1. CALCULATE 1-5 SCORES FOR R, F, M
# -------------------------------------------------------------------------
# Recency: Lower days = Higher Score (5 = most recent)
rfm['R_score'] = pd.qcut(rfm['recency_days'], q=5, labels=[5, 4, 3, 2, 1]).astype(
    int
)

# Frequency & Monetary: Higher value = Higher Score (5 = best)
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]
).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(
    int
)

# -------------------------------------------------------------------------
# 2. APPLY WEIGHTED SCORE (Monetary weighted 2x)
# Score Range: Min = 1+1+(2*1) = 4 | Max = 5+5+(2*5) = 20
# Scale/Normalize score to match the required 3-15 range
# -------------------------------------------------------------------------
raw_weighted_score = rfm['R_score'] + rfm['F_score'] + (2 * rfm['M_score'])

# Scaling min=4, max=20 linearly into min=3, max=15
rfm['Weighted_RFM_Score'] = np.round(
    3 + (raw_weighted_score - 4) * (15 - 3) / (20 - 4)
).astype(int)


# -------------------------------------------------------------------------
# 3. MAP GRANULAR SEGMENTS & STRATEGIC ACTIONS
# -------------------------------------------------------------------------
def map_advanced_segment(score):
  if score >= 13:
    return 'VIP Champions'
  elif score >= 11:
    return 'Loyal Customers'
  elif score >= 9:
    return 'Regular Customers'
  elif score >= 7:
    return 'New Customers'
  elif score >= 5:
    return 'At Risk'
  else:
    return 'Lost'


def map_action(segment):
  actions = {
      'VIP Champions': 'Early access, VIP discount codes',
      'Loyal Customers': 'Loyalty programme, rewards',
      'Regular Customers': 'Cross-sell campaigns',
      'New Customers': 'Onboarding, welcome series',
      'At Risk': 'Win-back campaign, personalised email',
      'Lost': 'Last attempt, big discount',
  }
  return actions.get(segment, '')


rfm['Advanced_Segment'] = rfm['Weighted_RFM_Score'].apply(map_advanced_segment)
rfm['Recommended_Action'] = rfm['Advanced_Segment'].apply(map_action)

# -------------------------------------------------------------------------
# 4. DISPLAY SUMMARY TABLE
# -------------------------------------------------------------------------
segment_order = [
    'VIP Champions',
    'Loyal Customers',
    'Regular Customers',
    'New Customers',
    'At Risk',
    'Lost',
]

advanced_summary = (
    rfm.groupby('Advanced_Segment')
    .agg(
        Customer_Count=('monetary', 'count'),
        Total_Revenue=('monetary', 'sum'),
        Avg_Spend=('monetary', 'mean'),
        Avg_Recency=('recency_days', 'mean'),
    )
    .reindex(segment_order)
    .reset_index()
)

print('=== ADVANCED RFM SUMMARY ===')
print(advanced_summary)

=== ADVANCED RFM SUMMARY ===
    Advanced_Segment  Customer_Count  Total_Revenue    Avg_Spend  Avg_Recency
0      VIP Champions             516     1280902.39  2482.368973    70.180233
1    Loyal Customers             305      392033.46  1285.355607   141.393443
2  Regular Customers             533      501101.52   940.152946   192.253283
3      New Customers             493      303115.51   614.838763   221.811359
4            At Risk             356      126511.07   355.368174   226.497191
5               Lost             337       73186.59   217.170890   389.427300


In [35]:
# -------------------------------------------------------------------------
# EXPORT TO CSV FOR MARKETING
# -------------------------------------------------------------------------
export_columns = [
    'recency_days',
    'frequency',
    'monetary',
    'R_score',
    'F_score',
    'M_score',
    'Weighted_RFM_Score',
    'Advanced_Segment',
    'Recommended_Action',
]

rfm[export_columns].to_csv('rfm_segments.csv')
print(' Successfully generated and exported "rfm_segments.csv"')

 Successfully generated and exported "rfm_segments.csv"


## Synthesis & Team Reflection

### 1. What was the biggest surprise?
The biggest surprise was the extreme revenue concentration within the **VIP Champions** segment, where just 516 customers (20.3% of the base) account for **€1,280,902.39**—nearly 48% of total revenue. We expected total spend to be far more evenly distributed across segment tiers rather than concentrated so heavily in top buyers.

### 2. What's our recommendation to Marko?
Marko should prioritize launching an automated re-engagement campaign for the **356 At-Risk customers** immediately, as their average recency has already reached 226 days. Re-engaging these high-potential past buyers represents the most cost-effective and urgent opportunity to recover high-margin revenue before they lapse into the Lost tier.

### 3. What data did we lack?
Our analysis was limited by the lack of product return records, which could artificially inflate the net value of high-spending buyers who frequently return items. Furthermore, incorporating website browsing behavioral logs and NPS customer satisfaction scores would enable us to flag declining customer interest much earlier.

In [33]:
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.
